# Denoising autoencoder con ruido gaussiano, Mirai

Añade una capa GaussianNoise a la entrada del autoencoder denso, activa solo en entrenamiento, y compara varios niveles de ruido con un control sin corrupción (nivel 0). La dirección IP de origen (`src_ip`) no se usa como entrada, porque identifica a los equipos atacantes.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path("../../data/flows.csv")

df = pd.read_csv(DATA_PATH).drop(columns=['class_name', 'flow_start'])
print(df.shape)

(1006776, 11)


In [3]:
from sklearn.model_selection import train_test_split

y = df['label'].values
X = pd.get_dummies(df.drop(columns=['label', 'src_ip']), columns=['state'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
X_train_normal = X_train[y_train == 0]

print(f"X_train: {X_train.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test.shape}")

X_train: (805420, 15), X_train_normal: (6234, 15), X_test: (201356, 15)


In [4]:
def build_dae(input_dim, hidden_units, noise_stddev):
    inputs = keras.Input(shape=(input_dim,))
    x = layers.GaussianNoise(noise_stddev)(inputs)
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)


HIDDEN_UNITS = [64, 32, 16]
NOISE_LEVELS = [0.0, 0.05, 0.1, 0.2]  # 0.0 = control sin ruido

results = {}
for noise in NOISE_LEVELS:
    tf.keras.utils.set_random_seed(SEED)
    model = build_dae(X_train.shape[1], HIDDEN_UNITS, noise)
    history = model.fit(
        X_train_normal, X_train_normal,
        epochs=100,
        batch_size=256,
        validation_split=0.1,
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
        verbose=0,
    )
    auc = roc_auc_score(y_test, reconstruction_error(model, X_test))
    results[noise] = {'model': model, 'auc': auc, 'epochs': len(history.history['loss'])}
    print(f"noise_stddev={noise}: epocas={results[noise]['epochs']}, ROC-AUC={auc:.4f}")

noise_stddev=0.0: epocas=100, ROC-AUC=0.9442


noise_stddev=0.05: epocas=74, ROC-AUC=0.8765


noise_stddev=0.1: epocas=56, ROC-AUC=0.9340


noise_stddev=0.2: epocas=46, ROC-AUC=0.8765


In [5]:
best_noise = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_noise]['model']
print(f"Mejor variante: noise_stddev={best_noise} (ROC-AUC={results[best_noise]['auc']:.4f})")
print(f"Control (noise_stddev=0.0, mismo codigo, sin ruido): ROC-AUC={results[0.0]['auc']:.4f}")
print(f"Diferencia mejor DAE vs control: {results[best_noise]['auc'] - results[0.0]['auc']:+.4f}")

threshold = np.percentile(reconstruction_error(best_model, X_train_normal), 95)
y_pred = (reconstruction_error(best_model, X_test) > threshold).astype(int)

print()
print(classification_report(y_test, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC (mejor variante DAE): {results[best_noise]['auc']:.4f}")

Mejor variante: noise_stddev=0.0 (ROC-AUC=0.9442)
Control (noise_stddev=0.0, mismo codigo, sin ruido): ROC-AUC=0.9442
Diferencia mejor DAE vs control: +0.0000



              precision    recall  f1-score   support

      normal       0.04      0.95      0.08      1559
      ataque       1.00      0.84      0.91    199797

    accuracy                           0.84    201356
   macro avg       0.52      0.89      0.50    201356
weighted avg       0.99      0.84      0.90    201356

[[  1479     80]
 [ 32550 167247]]
ROC-AUC (mejor variante DAE): 0.9442
